# SingBERT Scam / Non-Scam Classifier

This notebook fine-tunes and evaluates the classifier using the processed dataset saved at `datasets/combined_dataset.csv`. Run `dataset_processing.ipynb` first if you need to rebuild the merged dataset artifact.


## Environment Setup

Install the notebook dependencies and import the libraries used for training and evaluation.


In [ ]:
# Install only the extra packages needed in Colab.
# Leave Colab's torch / torchvision versions alone.
# Reinstalling one without the other can break CUDA support.
%pip install -q transformers datasets accelerate

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


## Project Configuration

Define the model checkpoint, artifact locations, and training hyperparameters.


In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "datasets"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
MODEL_OUTPUT_DIR = ARTIFACT_DIR / "singbert_classifier"
METRICS_OUTPUT_DIR = ARTIFACT_DIR / "metrics"

COMBINED_INPUT_PATH = DATA_DIR / "combined_dataset.csv"

EXPECTED_COLUMNS = ["text", "label", "persona", "source_type", "scenario"]
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

MODEL_CHECKPOINT = "zanelim/singbert"

RANDOM_SEED = 42
TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15

# SingBERT's published continued-pretraining used max_seq_length=128,
# so the default classifier setup keeps the same sequence length.
MAX_LENGTH = 128
NUM_EPOCHS = 4
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 160)


## Modeling Dataset Preparation

Load the processed CSV, validate the required columns, create numeric labels, and split the data for training.


In [ ]:
# Reload the processed dataset
model_df = pd.read_csv(COMBINED_INPUT_PATH)

required_columns = {TEXT_COLUMN, LABEL_COLUMN}
missing_required = required_columns - set(model_df.columns)
if missing_required:
    raise ValueError(f"The combined CSV is missing required columns: {sorted(missing_required)}")

for optional_column in ["persona", "source_type", "scenario", "source_file"]:
    if optional_column not in model_df.columns:
        model_df[optional_column] = "unknown"

model_df[TEXT_COLUMN] = model_df[TEXT_COLUMN].astype(str).str.strip()
model_df[LABEL_COLUMN] = model_df[LABEL_COLUMN].astype(str).str.strip()

dedupe_columns = [col for col in EXPECTED_COLUMNS if col in model_df.columns]
model_df = model_df.loc[
    model_df[TEXT_COLUMN].ne("") & model_df[LABEL_COLUMN].ne("")
].drop_duplicates(subset=dedupe_columns).reset_index(drop=True)

# Map string labels to numeric IDs
label_names = sorted(model_df[LABEL_COLUMN].unique().tolist())
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}
model_df["labels"] = model_df[LABEL_COLUMN].map(label2id)

In [ ]:
# Make train / validation / test splits
train_val_df, test_df = train_test_split(
    model_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=model_df["labels"],
)

# Split the remaining data into train and validation
validation_relative_size = VALIDATION_SIZE / (1 - TEST_SIZE)
train_df, validation_df = train_test_split(
    train_val_df,
    test_size=validation_relative_size,
    random_state=RANDOM_SEED,
    stratify=train_val_df["labels"],
)

split_frames = {"train": train_df, "validation": validation_df, "test": test_df}
for split_name, split_df in split_frames.items():
    print(f"{split_name:>10}: {len(split_df):4d} rows")
    print(split_df[LABEL_COLUMN].value_counts().sort_index().to_string())
    print()

# Convert the pandas splits to a Hugging Face DatasetDict
dataset_dict = DatasetDict(
    {
        split_name: Dataset.from_pandas(split_df.reset_index(drop=True), preserve_index=False)
        for split_name, split_df in split_frames.items()
    }
)

## Tokenization Setup

Convert each split into Hugging Face datasets, tokenize the text, and prepare padded PyTorch batches for the trainer.


In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )

# Tokenize all splits
tokenized_datasets = dataset_dict.map(tokenize_batch, batched=True)

tensor_columns = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in tokenized_datasets["train"].column_names:
    tensor_columns.append("token_type_ids")

# Return torch tensors for model inputs
tokenized_datasets.set_format(type="torch", columns=tensor_columns)

# Pad each batch just before it is fed to the model
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

# Save the label mapping for inference
label_mapping_path = METRICS_OUTPUT_DIR / "label_mapping.json"
with open(label_mapping_path, "w", encoding="utf-8") as fp:
    json.dump(
        {
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()},
        },
        fp,
        indent=2,
    )

print(f"Saved label mapping to: {label_mapping_path}")


## Training Pipeline

Define shared classification metrics, fine-tune the model, and then evaluate every split for reporting.


In [ ]:
def calculate_classification_metrics(true_ids, predicted_ids):
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        true_ids, predicted_ids, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        true_ids, predicted_ids, average="weighted", zero_division=0
    )

    return {
        "accuracy": accuracy_score(true_ids, predicted_ids),
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return calculate_classification_metrics(labels, predictions)

In [ ]:
# Load the classification model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)

# Training arguments
training_args_kwargs = dict(
    output_dir=str(MODEL_OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    seed=RANDOM_SEED,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

# Support both newer and older Transformers argument names
if "eval_strategy" in TrainingArguments.__dataclass_fields__:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

# Build the TrainingArguments object
training_args = TrainingArguments(**training_args_kwargs)

# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Fine-tune the model
train_result = trainer.train()

# Save the trained model and tokenizer
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

### Evaluation Helpers

Keep the evaluation code reusable by centralizing metric cleanup, per-split prediction, and classification report generation.


In [ ]:
# Convert NumPy scalars to plain Python values
def to_python_scalars(metrics_dict):
    cleaned = {}
    for key, value in metrics_dict.items():
        if isinstance(value, (np.floating, np.integer)):
            cleaned[key] = value.item()
        else:
            cleaned[key] = value
    return cleaned

def evaluate_split(split_name, dataset_split):
    prediction_output = trainer.predict(dataset_split, metric_key_prefix=split_name)
    predicted_ids = prediction_output.predictions.argmax(axis=-1)
    true_ids = prediction_output.label_ids

    split_metrics = to_python_scalars(
        {
            "loss": prediction_output.metrics.get(f"{split_name}_loss"),
            **calculate_classification_metrics(true_ids, predicted_ids),
        }
    )

    # Save a per-class report for this split
    report = classification_report(
        true_ids,
        predicted_ids,
        labels=list(range(len(label2id))),
        target_names=[id2label[idx] for idx in range(len(label2id))],
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose()
    report_path = METRICS_OUTPUT_DIR / f"{split_name}_classification_report.csv"
    report_df.to_csv(report_path)

    return split_metrics, report_df

In [ ]:
# Evaluate all dataset splits
validation_metrics, validation_report_df = evaluate_split("validation", tokenized_datasets["validation"])
train_metrics, train_report_df = evaluate_split("train", tokenized_datasets["train"])
test_metrics, test_report_df = evaluate_split("test", tokenized_datasets["test"])

summary_metrics = {
    "model_checkpoint": MODEL_CHECKPOINT,
    "combined_input_path": str(COMBINED_INPUT_PATH),
    "output_model_dir": str(MODEL_OUTPUT_DIR),
    "split_sizes": {split_name: len(split_df) for split_name, split_df in split_frames.items()},
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "train_runtime": to_python_scalars(train_result.metrics),
    "validation": validation_metrics,
    "train": train_metrics,
    "test": test_metrics,
}

# Save the summary metrics
summary_metrics_path = METRICS_OUTPUT_DIR / "summary_metrics.json"
with open(summary_metrics_path, "w", encoding="utf-8") as fp:
    json.dump(summary_metrics, fp, indent=2)

trainer_log_path = METRICS_OUTPUT_DIR / "trainer_log_history.json"
with open(trainer_log_path, "w", encoding="utf-8") as fp:
    json.dump([to_python_scalars(item) for item in trainer.state.log_history], fp, indent=2)

metrics_overview = pd.DataFrame([train_metrics, test_metrics], index=["train", "test"])
display(metrics_overview)
display(test_report_df)

print(f"Saved fine-tuned model to: {MODEL_OUTPUT_DIR}")
print(f"Saved metric summary to: {summary_metrics_path}")
print(f"Saved trainer history to: {trainer_log_path}")
print(f"Saved per-class reports to: {METRICS_OUTPUT_DIR}")
